# 1. What you'll learn

        - visualize the forward Gaussian noising schedule
- train a tiny time-conditioned noise predictor
- run and deliberately truncate reverse denoising

        **The one question this notebook answers:** How does gradual noising create a supervised denoising task, and why does reverse sampling need every step?

# 2. Setup

This lesson keeps diffusion basics small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import torch
from torch import nn
from sklearn.datasets import load_digits

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Real 8×8 digits scaled to [-1,1] allow a 30-step diffusion process. The tiny MLP is a mechanism demonstration, not a production image generator.

**Small example:** With clean x=1, cumulative signal ᾱ=.64, and noise ε=-.5, xt=√.64×1+√.36×(-.5)=.8-.3=.5. Training reveals ε so the network learns to predict it.

In [ ]:
d=load_digits();X=torch.tensor((d.data/8-1).astype("float32"));labels=d.target;X=X[:1500];print("Images:",X.shape,"range:",(X.min().item(),X.max().item()),"classes:",np.bincount(labels[:1500]).tolist())

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
T=30;betas=torch.linspace(.0008,.12,T);alphas=1-betas;abar=torch.cumprod(alphas,0);idx=0;eps=torch.randn(64,generator=torch.Generator().manual_seed(SEED));noisy=[]
for step in [0,7,15,29]: noisy.append((abar[step].sqrt()*X[idx]+(1-abar[step]).sqrt()*eps).reshape(8,8))
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].imshow(np.hstack(noisy),cmap="gray");axes[0].axis("off");axes[0].set_title("Forward steps 0, 7, 15, 29")
axes[1].plot(abar);axes[1].set(title="Remaining signal",xlabel="step",ylabel="cumulative alpha");axes[2].plot(torch.sqrt(1-abar));axes[2].set(title="Noise standard deviation",xlabel="step");plt.tight_layout();plt.show()

**Takeaway:** Forward noising gradually erases digit structure.

**Takeaway:** Cumulative alpha measures the clean-signal fraction left.

**Takeaway:** Late steps are dominated by noise, so denoising must use the time index.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
torch.manual_seed(SEED);time_embed=nn.Embedding(T,16);denoiser=nn.Sequential(nn.Linear(64+16,96),nn.SiLU(),nn.Linear(96,96),nn.SiLU(),nn.Linear(96,64));params=list(time_embed.parameters())+list(denoiser.parameters());opt=torch.optim.Adam(params,lr=.002);print("Denoiser parameters:",sum(p.numel() for p in params))

# 6. Train

        Training turns the assumptions behind diffusion basics into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Choose a random diffusion time for each clean image.
2. Generate xt directly from clean x0 and sampled Gaussian noise.
3. Predict the exact noise using xt plus a learned time embedding.
4. During sampling, repeatedly subtract predicted noise from T−1 down to 0.

In [ ]:
losses=[];rng=torch.Generator().manual_seed(SEED)
for step in range(700):
 ids=torch.randint(0,len(X),(96,),generator=rng);t=torch.randint(0,T,(96,),generator=rng);clean=X[ids];noise=torch.randn(clean.shape,generator=rng);a=abar[t,None];xt=a.sqrt()*clean+(1-a).sqrt()*noise;pred=denoiser(torch.cat([xt,time_embed(t)],1));loss=nn.functional.mse_loss(pred,noise);opt.zero_grad();loss.backward();opt.step();losses.append(loss.item())
print("Noise MSE first/last 50:",round(np.mean(losses[:50]),3),round(np.mean(losses[-50:]),3))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
@torch.no_grad()
def sample(stop=0,n=64):
 x=torch.randn(n,64,generator=torch.Generator().manual_seed(SEED))
 for ti in range(T-1,stop-1,-1):
  t=torch.full((n,),ti,dtype=torch.long);pred=denoiser(torch.cat([x,time_embed(t)],1));a=alphas[ti];ab=abar[ti];x=(x-(1-a)/torch.sqrt(1-ab)*pred)/torch.sqrt(a)
  if ti>0: x+=torch.sqrt(betas[ti])*torch.randn(x.shape,generator=torch.Generator().manual_seed(SEED+ti))
 return x.clamp(-1,1)
samples=sample();nearest=torch.cdist(samples,X[::5]).min(1).values.mean().item();noise_base=torch.cdist(torch.randn_like(samples),X[::5]).min(1).values.mean().item()
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(losses);axes[0].set_title("Noise-prediction loss")
axes[1].imshow(np.block([[samples[r*8+c].reshape(8,8).numpy() for c in range(8)] for r in range(8)]),cmap="gray");axes[1].axis("off");axes[1].set_title("Tiny reverse samples")
axes[2].bar(["raw noise","reverse samples"],[noise_base,nearest],color=[".6","#176b66"]);axes[2].set_title("Nearest-real Euclidean distance");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print(f"Nearest-real distance samples={nearest:.2f}; raw-noise baseline={noise_base:.2f}")

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Reverse generation is a chain; skipping the later denoising steps leaves significant noise and compounds approximation errors. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
truncated=sample(stop=15);trunc_nearest=torch.cdist(truncated,X[::5]).min(1).values.mean().item();print(f"Full 30-step nearest-real distance={nearest:.2f}; stopping halfway={trunc_nearest:.2f}; remaining roughness variance={truncated.var().item():.3f}")

**Use this when…** stable high-quality generative modelling justifies iterative sampling cost.

        **Don't use this when…** you need instant generation, tiny compute, or this educational MLP is being mistaken for a production diffusion model.

        ## Try this

        1. Use 10 versus 60 diffusion steps.
2. Plot predicted versus true noise MSE by time.
3. Condition the denoiser on digit labels.